# Pipeline Orchestrator
This notebook acts as the orchestrator for the modularized sarcasm detection pipeline.
Run the cell that sequentially executes the full workflow for all models.

In [ ]:
import sys, os
import torch, gc

# Fix paths
sys.path.append(os.path.abspath('..'))
if os.path.basename(os.getcwd()) == 'utils':
    os.chdir('..')

# Imports
from utils.config import get_config
from utils.environment import setup_environment
from utils.data_loading import load_and_prepare_data
from utils.tokenization import load_tokenizer_and_tokenize
from utils.model_setup import load_base_model, get_lora_config
from utils.training import train_all_adapters
from utils.combined_adapter import run_combined_experiment
from utils.evaluation import evaluate_cross_variety
from utils.zero_shot import run_zero_shot_baseline
from utils.results import generate_all_results
from utils.error_export import export_error_analysis

# Config handling (controlled from main.ipynb)
DEFAULT_CONFIGS = ['tinyllama_lora', 'qwen25_lora', 'xlmr', 'xlmr_lora']

if 'CONFIG_NAMES' not in globals():
    CONFIG_NAMES = DEFAULT_CONFIGS

print(f"\n[models.ipynb] Running configs: {CONFIG_NAMES}\n")


# Main loop
for config_name in CONFIG_NAMES:
    print('\n' + '='*60)
    print(f'Starting Pipeline: {config_name}')
    print('='*60 + '\n')

    # 1. Setup Configuration
    CONFIG = get_config(config_name)
    setup_environment(CONFIG)

    # 2. Load & Prepare Data
    print('\nStarting Data Loading & Preparation')
    variety_data, raw_datasets = load_and_prepare_data(CONFIG)
    print('Finished Data Preparation\n')

    # 3. Tokenization
    print('\nStarting Tokenization')
    tokenizer, tokenized = load_tokenizer_and_tokenize(CONFIG, variety_data)
    print('Finished Tokenization\n')

    # 4. Model Setup
    print('\nStarting Model Initialization')
    base_model, total_params = load_base_model(CONFIG, tokenizer)
    lora_cfg = get_lora_config(CONFIG)
    print('Finished Model Initialization\n')

    # 5. Training
    print('\nStarting Main Training Loop')
    adapter_registry, training_log = train_all_adapters(
        CONFIG, tokenizer, tokenized, lora_cfg
    )
    print('Finished Main Training Loop\n')

    # 6. Combined Adapter Experiment
    print('\nStarting Combined Adapter Experiment')
    combined_eval_results = run_combined_experiment(
        CONFIG, tokenizer, tokenized, lora_cfg, adapter_registry
    )
    print('Finished Combined Adapter Experiment\n')

    # 7. Cross-Variety Evaluation
    print('\nStarting Cross-Variety Evaluation')
    eval_results = evaluate_cross_variety(
        CONFIG, tokenizer, tokenized, adapter_registry
    )
    print('Finished Cross-Variety Evaluation\n')

    # 8. Zero-Shot Baseline
    print('\nStarting Zero-Shot Baseline')
    zs_results = run_zero_shot_baseline(CONFIG, tokenizer, tokenized)
    print('Finished Zero-Shot Baseline\n')

    # 9. Results & Visualization
    print('\nStarting Results Generation')
    generate_all_results(
        CONFIG,
        eval_results,
        combined_eval_results,
        adapter_registry,
        training_log,
        total_params,
        zs_results=zs_results
    )
    print('Finished Results Generation\n')

    # 10. Error Export
    print('\nStarting Error Export')
    export_error_analysis(
        CONFIG,
        eval_results,
        combined_eval_results,
        tokenized,
        adapter_registry,
        tokenizer,
        variety_data
    )
    print('Finished Error Export\n')

    print('\n' + '='*60)
    print(f'PIPELINE COMPLETE FOR: {config_name}')
    print('='*60 + '\n')

    # Cleanup
    del base_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()